# ➰ Plotting Probabilities of Corrupted Model Runs ➰

## Parameters

In [1]:
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"

oneshot_template = ""
last_prompt_template = ""
num_shots = 1

data_path = "data/noun-adj.csv"
src_lang_base = "eng"
tgt_lang_base = "ger"
src_lang_source = "eng"
tgt_lang_source = "ger"

sample_size = 50
random_seed = 42

sentences_src_prefix_base = "phrase-"
sentences_tgt_prefix_base = "phrase-"
sentences_src_prefix_source = "phrase_head_final-"
sentences_tgt_prefix_source = "phrase_head_final-"

noun_base_prefix = "noun-"
verb_base_prefix = "past_participle-"
noun_source_prefix = "noun-"
verb_source_prefix = "past_participle-"

load_data = False # CRUCIAL: if True, doesn't run intervention, just loads existing data


block_intervention_save_path = \
    f"output/intervention/block-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"

head_intervention_save_path = \
    f"output/intervention/head-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"

In [2]:
# Parameters
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"
oneshot_template = "{lang_src}: \"{sentence_src}\" - {lang_tgt}: \"{sentence_tgt}\""
last_prompt_template = "{lang_src}: \"{sentence_src}\" - {lang_tgt}: \"{sentence_tgt}"
num_shots = 1
data_path = "data/past-participle.csv"
src_lang_base = "eng"
tgt_lang_base = "fre"
src_lang_source = "eng"
tgt_lang_source = "ger"
sample_size = 150
random_seed = 42
sentences_src_prefix_base = "phrase_participle-"
sentences_tgt_prefix_base = "phrase_cutoff_after_auxiliary-"
sentences_src_prefix_source = "phrase_participle-"
sentences_tgt_prefix_source = "phrase_cutoff_after_auxiliary-"
noun_base_prefix = "object_noun-"
verb_base_prefix = "past_participle-"
noun_source_prefix = "object_noun-"
verb_source_prefix = "past_participle-"


In [3]:
print(tgt_lang_base)

fre


In [4]:
if block_intervention_save_path is None:
    block_intervention_save_path = \
        f"output/intervention/block-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"
if head_intervention_save_path is None:
    head_intervention_save_path = \
        f"output/intervention/head-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"
block_intervention_plot_save_path = block_intervention_save_path.replace(".csv", ".png")
head_intervention_plot_save_path = head_intervention_save_path.replace(".csv", ".png")


# TODO: don't hardcode
shot_data_src_base = f'phrase_participle-{src_lang_base}'
shot_data_tgt_base = f'phrase-{tgt_lang_base}'
shot_data_src_source = f'phrase_participle-{src_lang_source}'
shot_data_tgt_source = f'phrase_participle-{tgt_lang_source}'

## Setup

In [5]:
import torch
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from typing import List, Dict, Any
from transformers import AutoModelForCausalLM, AutoTokenizer

import pyvene as pv
from pyvene import embed_to_distrib, top_vals, format_token

from create_datasets.parallel_dataset import ParallelDataset

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
sm = torch.nn.Softmax(dim=2)

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


### Set up the Model

In [6]:
if not load_data:
    model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

### Set up the Data

In [7]:
if not load_data:
    df = pd.read_csv(data_path)

    base_df = df.sample(n=sample_size, random_state=random_seed).reset_index(drop=True)
    source_df = base_df.sample(n=sample_size, random_state=random_seed).reset_index(drop=True) # shuffled base

    base_dataset = ParallelDataset(
        model_id,
        dataframe=base_df,
        lang_src=src_lang_base,
        lang_tgt=tgt_lang_base,
        sentences_src_prefix=sentences_src_prefix_base,
        sentences_tgt_prefix=sentences_tgt_prefix_base,
        random_seed=random_seed,
    )

    source_dataset = ParallelDataset(
        model_id,
        dataframe=source_df,
        lang_src=src_lang_source,
        lang_tgt=tgt_lang_source,
        sentences_src_prefix=sentences_src_prefix_source,
        sentences_tgt_prefix=sentences_tgt_prefix_source,
        random_seed=random_seed,
    )

    base_prompts = base_dataset.format(
        oneshot_template,
        shots=num_shots,
        last_prompt_template=last_prompt_template,
        shot_data_src=shot_data_src_base,
        shot_data_tgt=shot_data_tgt_base,
        )
    print("=====Example of Base Prompt=====")
    print(base_prompts[0])

    base_tokens = base_dataset.prompts_to_tokens()
    base_last_token_indices = base_dataset.last_token_indices

    source_prompts = source_dataset.format(
        oneshot_template,
        shots=num_shots,
        last_prompt_template=last_prompt_template,
        shot_data_src=shot_data_src_source,
        shot_data_tgt=shot_data_tgt_source,
        )
    print("=====Example of Source Prompt=====")
    print(source_prompts[0])

    source_tokens = source_dataset.prompts_to_tokens()
    source_last_token_indices = source_dataset.last_token_indices

config.json:   0%|          | 0.00/738 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.45G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

Loaded pretrained model ai-forever/mGPT into HookedTransformer


Loaded pretrained model ai-forever/mGPT into HookedTransformer
=====Example of Base Prompt=====
English: "The knight has led the horse" - Français: "Le chevalier a mené le cheval"
English: "The man has dropped the bell" - Français: "L'homme a
=====Example of Source Prompt=====
English: "The thief has stolen the parcel" - Deutsch: "Der Dieb hat das Paket gestohlen"
English: "The dog has pulled the chain" - Deutsch: "Der Hund hat


### Functions

#### Intervention Config

In [8]:
def intervention_config(model_type, intervention_type, unit, layer):
    """
    Parameters
    __________

    model_type: model type
    intervention_type: component for RepresentationConfig, e.g. head_attention_value_output
    unit: string to define the component type, e.g. "h" (head), "pos" (position), "h.pos" (head within position)
    later: layer id
    """
    # Set up the config to intervene
    config = pv.IntervenableConfig(
        model_type=model_type,
        representations=[
            pv.RepresentationConfig(
                layer,  # layer
                intervention_type,  # intervention type
                unit,  # intervention unit is now [pos] within [h]
                1,  # max number of unit
            ),
        ],
        intervention_types=pv.VanillaIntervention,
    )
    return config

#### Intervention Function

In [ ]:
def intervention_data(
        base: torch.Tensor,
        source: torch.Tensor,
        base_pos: int,
        source_pos: int,
        tokentype2token: Dict[str, str],
        component_type: str,
        head_i: int = None,
        data: List[Dict[str, Any]] = None
    ) -> List[Dict[str, Any]]:
    """
    Collect intervention data for a given model component (block output or head attention value output).

    Parameters
    ----------
    base : torch.Tensor
        The tokenized base prompt tensor.
    source : torch.Tensor
        The tokenized source prompt tensor.
    base_pos : int
        The position of the last token in the base prompt.
    source_pos : int
        The position of the last token in the source prompt.
    tokentype2token : Dict[str, str]
        A dictionary mapping token types (e.g., 'noun-base', 'adj-base') to their corresponding token strings.
    component_type : str
        The type of model component to intervene on. Must be either 'block_output' or 'head_attention_value_output'.
    head_i : int, optional
        The index of the head to intervene on (if applicable). Only used for head-level interventions.
    data : List[Dict[str, Any]], optional
        A list to append the collected data to. If None, a new list will be created.

    Returns
    -------
    List[Dict[str, Any]]
        A list of dictionaries containing the intervention data, with keys:
        - "token_type": The type of token (e.g., 'noun-base', 'adj-base').
        - "token": The token string.
        - "prob": The probability of the token after intervention.
        - "layer": The layer index.
        - "head_i": The head index (if applicable).
        - "pos": The position index.
        - "type": The component type (e.g., 'block_output').

    Raises
    ------
    NotImplementedError
        If the component_type is not 'block_output' or 'head_attention_value_output'.
    """
    if data is None:
        data = []
    for layer_i in range(model.config.n_layer):
        if component_type == "block_output":
            unit = "pos"
        elif component_type == "head_attention_value_output":
            unit = "h.pos"
        else:
            raise NotImplementedError("Unsupported component type: {component_type}")
        config = intervention_config(
            type(model), component_type, unit, layer_i
        )
        intervenable = pv.IntervenableModel(config, model)
        if head_i is not None:
            unit_locations = {
                "sources->base": (
                    [[[[head_i]], [[source_pos]]]],  # intervene w/ target_head's pos_i
                    [[[[head_i]], [[base_pos]]]]
                ),
            }
        else:
            unit_locations = {"sources->base": (source_pos, base_pos)}
        # print(unit_locations)
        # print(base)
        # print(source)
        _, counterfactual_outputs = intervenable(
            base,
            source,
            unit_locations,
        )
        with torch.inference_mode():
            distrib = sm(counterfactual_outputs.logits)
        print(f"\nTOP VALUES AT LAYER #{layer_i} AT POSITION {base_pos}:")
        top_vals(tokenizer, distrib[0][base_pos], 5)
        for token_type, token in tokentype2token.items():
            data.append(
                {
                    "token_type": token_type,
                    "token": token,
                    "prob": float(distrib[0][base_pos][tokenizer.encode(token)[0],]),
                    "layer": layer_i,
                    "head_i": head_i,
                    "pos": base_pos,
                    "type": component_type,
                }
            )
    return data

## Block Output Intervention

### Calculating

In [10]:
if load_data:
    df = pd.read_csv(block_intervention_save_path)

else:
    data = []
    source_df_list = list(source_df.iterrows())
    for row_i, row in base_dataset.df.iterrows():
        # tokenize prompts
        prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)

        prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
        # last token index
        pos_base = prompt_base.input_ids.size(1) - 1
        pos_source = prompt_source.input_ids.size(1) - 1
        # token type to token dict
        tokentype2token = {
            # TODO: articles
            f"noun-base-{tgt_lang_base}": row[f'{noun_base_prefix}{tgt_lang_base}'],
            f"verb-base-{tgt_lang_base}": row[f'{verb_base_prefix}{tgt_lang_base}'],
            f"noun-source-{tgt_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_base}'],
            f"verb-source-{tgt_lang_base}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_base}'],

            f"noun-base-{tgt_lang_source}": row[f'{noun_base_prefix}{tgt_lang_source}'],
            f"verb-base-{tgt_lang_source}": row[f'{verb_base_prefix}{tgt_lang_source}'],
            f"noun-source-{tgt_lang_source}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_source}'],
            f"verb-source-{tgt_lang_source}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_source}'],
            
            f"art-base-{tgt_lang_base}": row[f'object_article-{tgt_lang_base}'],
            f"art-source-{tgt_lang_source}": source_df_list[row_i][1][f'object_article-{tgt_lang_source}'],
        }
        # print(tokentype2token)

        data = intervention_data(
            prompt_base, 
            prompt_source, 
            pos_base,
            pos_source,
            tokentype2token, 
            "block_output", 
            data=data
        )
    df = pd.DataFrame(data)
    df.to_csv(block_intervention_save_path)

### Plotting

In [11]:
# Create a line plot for token probabilities over layers
fig = px.line(
    df.groupby(['token_type', 'layer']).mean().reset_index(),
    x="layer",
    y="prob",
    color="token_type",
    title=f"Probabilities after Block Intervention ({src_lang_base}-{tgt_lang_base} base & {src_lang_source}-{tgt_lang_source} source)",
    labels={"layer": "Layer", "prob": "Probability", "token_type": "Token"},
    # category_orders={"layer": [str(i) for i in range(model.config.n_layer)]},
)

# Show the plot
fig.show()
# fig.write_image(block_intervention_plot_save_path)

/var/lib/condor/execute/dir_516941/ipykernel_106/655942255.py:3: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df.groupby(['token_type', 'layer']).mean().reset_index(),


## Head Intervention

### Calculating

In [12]:
# if load_data:
#     df = pd.read_csv(head_intervention_save_path)
# else:
#     data = []
#     source_df_list = list(source_df.iterrows())

#     for row_i, row in base_dataset.df.iterrows():
#         # tokenize prompts
#         prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)
#         prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
#         # last token index
#         pos_base = prompt_base.input_ids.size(1) - 1
#         pos_source = prompt_source.input_ids.size(1) - 1
#         # token type to token dict
#         tokentype2token = {
#             # TODO: articles
#             f"noun-base-{tgt_lang_base}": row[f'{noun_base_prefix}{tgt_lang_base}'],
#             f"verb-base-{tgt_lang_base}": row[f'{verb_base_prefix}{tgt_lang_base}'],
#             f"noun-source-{tgt_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_base}'],
#             f"verb-source-{tgt_lang_base}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_base}'],

#             f"noun-base-{tgt_lang_source}": row[f'{noun_base_prefix}{tgt_lang_source}'],
#             f"verb-base-{tgt_lang_source}": row[f'{verb_base_prefix}{tgt_lang_source}'],
#             f"noun-source-{tgt_lang_source}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_source}'],
#             f"verb-source-{tgt_lang_source}": source_df_list[row_i][1][f'{verb_source_prefix}{tgt_lang_source}'],

#             f"art-base-{tgt_lang_base}": row[f'object_article-{tgt_lang_base}'],
#             f"art-source-{tgt_lang_source}": source_df_list[row_i][1][f'object_article-{tgt_lang_source}'],
#         }

#         for head_i in range(model.config.n_head):
#             data = intervention_data(
#                 prompt_base, 
#                 prompt_source, 
#                 pos_base,
#                 pos_source,
#                 tokentype2token, 
#                 "head_attention_value_output", 
#                 head_i=head_i,
#                 data=data,
#             )
#     df = pd.DataFrame(data)
#     df.to_csv(head_intervention_save_path)

### Plotting

In [13]:
# import plotly.express as px
# import pandas as pd

# # Iterate over the unique tokens and create a heatmap for each
# tokens = df["token_type"].unique()
# for token in tokens:
#     token_df = df[df["token_type"] == token].groupby(['layer', 'head_i']).mean().reset_index()
#     heatmap_data = token_df.pivot(index="layer", columns="head_i", values="prob")

#     fig = px.imshow(
#         heatmap_data,
#         labels={"x": "Head", "y": "Layer", "color": "Probability"},
#         title=f"Probability Heatmap for Token after Head Intervention: {token}",
#         color_continuous_scale="viridis",
#     )
#     fig.show()

In [14]:
# HEAD 15.2 PREFERS NOUN-SOURCE MORE THAN ADJ-SOURCE?
# (difference not too big but it's interesting it is more visible. It sees the adjective too though.)

# Next up: Try with other languages as well (English-Italian, German-French)
# Bonus: Look into h.pos vs. pos: wtf are they doing?